# 01. Vector Product and Additive Quantization: VQ, PQ, RVQ, and AQLM

- **Category:** 14_quantisation / 04_vector_lattice_qat_and_lowbit
- **Lineage & Citations:** Jégou et al. (2011, *Product Quantization for Nearest Neighbor Search*); Babenko & Lempitsky (2014, *Additive Quantization for Extreme Vector Compression*); Egiazarian et al. (2024, *AQLM: Extreme Compression of Large Language Models via Additive Quantization*).
- **Purpose:** Exploit high-dimensional geometric packing gains and cross-weight correlations by grouping weights into multi-dimensional vectors and quantizing them jointly via single (VQ), subvector (PQ), residual (RVQ), and additive multi-codebook (AQLM) representations.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [07. SpQR and SqueezeLLM](../03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. Trellis and E8 Lattice Quantization QuIP# QTIP](02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use greedy uncoordinated residual vector quantization in AQLM without joint codebook fine-tuning (causes compounding residual drift); do NOT use full monolithic VQ when vector dimension $D > 8$ with high bit-rates (causes exponential codebook explosion $2^b$).

## Mathematical Foundations & High-Dimensional Geometry

1. **Space-Filling Gain (Shannon Rate-Distortion)**:
   $$\text{MSE}_{\text{vector}} < \text{MSE}_{\text{scalar}} \quad \text{at identical bit-rates } R = \frac{b}{d}$$
2. **Vector Quantization (VQ)**:
   $$q(\mathbf{w}) = \arg\min_{\mathbf{c}_k \in \mathcal{C}} \|\mathbf{w} - \mathbf{c}_k\|_2^2$$
3. **Product Quantization (PQ)**:
   $$\mathbf{w} = [\mathbf{w}^{(1)}, \dots, \mathbf{w}^{(M)}], \quad q_{\text{PQ}}(\mathbf{w}) = [q^{(1)}(\mathbf{w}^{(1)}), \dots, q^{(M)}(\mathbf{w}^{(M)})]$$
4. **Additive Multi-Codebook Quantization (AQLM)**:
   $$\mathbf{w} \approx \sum_{m=1}^M \mathcal{C}^{(m)}[i_m], \quad i_m \in \{0, \dots, K-1\}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def vector_quantize(vectors: torch.Tensor, codebook: torch.Tensor):
    dist = torch.sum(vectors**2, dim=1, keepdim=True) - 2.0 * (vectors @ codebook.T) + torch.sum(codebook**2, dim=1, keepdim=True).T
    indices = torch.argmin(dist, dim=1)
    return codebook[indices], indices

def product_quantize(vectors: torch.Tensor, codebooks: torch.Tensor):
    m, k, d_sub = codebooks.shape
    recons, idxs = [], []
    for i in range(m):
        r, idx = vector_quantize(vectors[:, i*d_sub:(i+1)*d_sub], codebooks[i])
        recons.append(r); idxs.append(idx.unsqueeze(1))
    return torch.cat(recons, dim=1), torch.cat(idxs, dim=1)

def residual_vector_quantize(vectors: torch.Tensor, codebooks: torch.Tensor):
    res = vectors.clone(); recon = torch.zeros_like(vectors); idxs = []
    for i in range(codebooks.shape[0]):
        r, idx = vector_quantize(res, codebooks[i])
        recon += r; res -= r; idxs.append(idx.unsqueeze(1))
    return recon, torch.cat(idxs, dim=1)

def aqlm_forward(x: torch.Tensor, indices: torch.Tensor, codebooks: torch.Tensor, d_in: int, d_out: int, bias: torch.Tensor | None = None):
    recon = sum(codebooks[i][indices[:, i].long()] for i in range(indices.shape[1]))
    w_deq = recon.reshape(d_in, d_out)
    y = x @ w_deq
    return ((y + bias) if bias is not None else y), w_deq

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. VQ, PQ, RVQ parity
vq_r, vq_i = vector_quantize(weights["w_vecs"], weights["vq_codebook"])
assert float(torch.max(torch.abs(vq_r - weights["vq_recon"]))) < 1e-5
assert float(torch.max(torch.abs(vq_i.float() - weights["vq_indices"]))) < 1e-5

pq_r, pq_i = product_quantize(weights["w_vecs"], weights["pq_codebooks"])
assert float(torch.max(torch.abs(pq_r - weights["pq_recon"]))) < 1e-5
assert float(torch.max(torch.abs(pq_i.float() - weights["pq_indices"]))) < 1e-5

rvq_r, rvq_i = residual_vector_quantize(weights["w_vecs"], weights["rvq_codebooks"])
assert float(torch.max(torch.abs(rvq_r - weights["rvq_recon"]))) < 1e-5
assert float(torch.max(torch.abs(rvq_i.float() - weights["rvq_indices"]))) < 1e-5

# 3. AQLM forward parity
out_aqlm, w_aqlm_deq = aqlm_forward(weights["x_calib"], weights["aqlm_indices"], weights["aqlm_codebooks"], 16, 8, weights["b_orig"])
assert float(torch.max(torch.abs(w_aqlm_deq - weights["w_aqlm_deq"]))) < 1e-5
assert float(torch.max(torch.abs(out_aqlm - weights["out_aqlm"]))) < 1e-5
print("All VQ, PQ, RVQ, and AQLM parity assertions verified against golden checkpoint!")

In [ ]:
# Comparative SQNR Benchmark: VQ vs PQ vs RVQ vs AQLM
w_target = weights["w_vecs"]
sig_pwr = float(torch.mean(w_target ** 2))

def eval_sqnr(w_recon: torch.Tensor) -> float:
    mse = float(torch.mean((w_target - w_recon) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_vq = eval_sqnr(weights["vq_recon"])
sqnr_pq = eval_sqnr(weights["pq_recon"])
sqnr_rvq = eval_sqnr(weights["rvq_recon"])

print(f"Single Codebook VQ SQNR: {sqnr_vq:.2f} dB (0.75 bits/weight)")
print(f"Product Quant (PQ) SQNR:  {sqnr_pq:.2f} dB (1.50 bits/weight)")
print(f"Residual Quant (RVQ) SQNR:{sqnr_rvq:.2f} dB (1.50 bits/weight)")
assert sqnr_rvq > sqnr_vq, "RVQ multi-stage refinement must exceed single VQ SQNR!"
assert sqnr_rvq > 3.0, "RVQ must achieve positive reconstruction SQNR!"

In [ ]:
# Latency Microbenchmark: Fused AQLM Multi-Codebook Table Lookup
n_vecs, d = 32768, 8
indices_mock = torch.randint(0, 256, (n_vecs, 2))
codebooks_mock = torch.randn(2, 256, d) * 0.1

t0 = time.perf_counter()
for _ in range(50):
    w_rec = codebooks_mock[0][indices_mock[:, 0]] + codebooks_mock[1][indices_mock[:, 1]]
t_aqlm = (time.perf_counter() - t0) / 50 * 1e3

print(f"Fused AQLM Table Lookup (32768x8) latency: {t_aqlm:.3f} ms")
assert t_aqlm < 20.0, "AQLM table lookup dequantization must execute in <20 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Space-Filling Gain**: Jointly quantizing multi-dimensional weight vectors captures sphere-packing geometric gains that scalar rounding cannot access.
2. **Product Subspaces**: PQ splits vectors into orthogonal subspaces, representing $K^M$ virtual centroids with only $M \times K$ parameters.
3. **Additive Synthesis**: AQLM uses multiple codebooks additively ($W = \sum C^{(m)}[i_m]$), enabling 2-bit weight representation with near-FP16 fidelity.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.md](../../../markdown/14_quantisation/04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.md)
- **PyTorch 2.x Module:** [01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.py](../../../pytorch2/14_quantisation/04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.py)
- **Functional JAX Module:** [01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.py](../../../jax/14_quantisation/04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.py)
- **Native Rust Candle Track:** [01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.rs](../../../candle/14_quantisation/04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.rs)
- **Hardware Kernel (CPU / WGSL):** [01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.rs](../../../kernels/14_quantisation/04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.rs)
- **Golden SafeTensors:** [01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.safetensors](../../../weights/14_quantisation/04_vector_lattice_qat_and_lowbit/01_vector_product_and_additive_quantization_vq_pq_rvq_aqlm.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 07. SpQR and SqueezeLLM](../03_llm_ptq_algorithms/07_spqr_and_squeezellm_sparse_plus_nonuniform_ptq.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 02. Trellis and E8 Lattice Quantization QuIP# QTIP](02_trellis_and_e8_lattice_quantization_quip_sharp_qtip.ipynb) &rarr;